# Questões faltantes da atividade 2

Este notebook não refaz a entrega conjunta. Ele só responde as cinco perguntas que ficaram vazias no final da Aula 2.

A Tabela 1.1.1 traz o número médio de horas semanais em cuidados de pessoas e afazeres domésticos, por sexo e cor ou raça, em 2022. A coluna usada é “preta ou parda”. Ela não separa pessoas pretas de pessoas pardas.

Execute a partir da pasta da atividade ou da raiz do repositório `analise_dados`.


In [1]:
import collections
import collections.abc

for nome in ("Iterable", "Mapping", "Sequence"):
    if not hasattr(collections, nome):
        setattr(collections, nome, getattr(collections.abc, nome))

from pathlib import Path

import pandas as pd
import xlrd

def ler_xls(caminho, aba):
    # xlrd 1.2 abre .xls. O pandas recente pede xlrd 2, que não lê esse formato.
    livro = xlrd.open_workbook(str(caminho))
    planilha = livro.sheet_by_name(aba)
    linhas = [planilha.row_values(i) for i in range(planilha.nrows)]
    return pd.DataFrame(linhas)

def raiz_do_repositorio():
    aqui = Path.cwd().resolve()
    for pasta in [aqui, *aqui.parents]:
        if (pasta / "dados").is_dir() and (pasta / "etapa_3").is_dir():
            return pasta
    raise FileNotFoundError("Execute a partir do repositorio analise_dados.")

DADOS = raiz_do_repositorio() / "dados"
arquivo = DADOS / "Tabela 1.1.1.xls"
arquivo.exists()


True

In [2]:
bruto = ler_xls(arquivo, "2022")
print("abas do arquivo incluem 2022 e anos anteriores. shape da aba 2022:", bruto.shape)
print(bruto.iloc[0, 0])

indicador_1 = bruto.iloc[8:41].copy()
indicador_1.columns = [
    "uf_regiao",
    "total",
    "total_branca",
    "total_preta_parda",
    "homem_branca",
    "homem_preta_parda",
    "mulher_branca",
    "mulher_preta_parda",
]
indicador_1 = indicador_1.reset_index(drop=True)
for coluna in indicador_1.columns[1:]:
    indicador_1[coluna] = pd.to_numeric(indicador_1[coluna], errors="coerce")

indicador_1


abas do arquivo incluem 2022 e anos anteriores. shape da aba 2022: (46, 8)
Tabela 1.1.1 - Número médio de horas semanais dedicadas aos cuidados de pessoas e/ou afazeres domésticos das pessoas de 14 anos ou mais de idade, na semana de referência, por sexo e cor ou raça, segundo Grandes Regiões e Unidades da Federação - Brasil - 2022


,uf_regiao,total,total_branca,total_preta_parda,homem_branca,homem_preta_parda,mulher_branca,mulher_preta_parda
0,Brasil,16.985781,16.545930,17.333142,11.705136,11.746330,20.404106,22.037998
1,Norte,16.187037,15.925246,16.256032,11.588653,11.514646,19.295780,20.534188
2,Rondônia,17.014162,16.727733,17.101523,12.111127,12.544080,20.412887,21.061410
3,Acre,13.606903,13.806929,13.541399,10.796120,9.686983,16.130352,16.710754
4,Amazonas,16.807290,15.987545,16.995493,12.233043,12.822584,18.764969,20.716986
5,Roraima,13.639257,13.651755,13.633606,10.814803,10.476601,16.135877,16.560244
6,Pará,16.474268,16.181627,16.560239,11.524306,11.026707,19.835110,21.591144
7,Amapá,13.586656,14.389180,13.415091,11.116821,10.749444,16.648579,15.908487
8,Tocantins,15.815781,15.701788,15.768728,10.733863,11.682616,19.663713,19.629399
9,Nordeste,18.499396,18.151700,18.610911,11.723907,11.848436,22.744666,23.716716


In [3]:
print("linhas:", len(indicador_1))
print("ausentes:", int(indicador_1.isna().sum().sum()))
print(indicador_1["uf_regiao"].tolist())


linhas: 33
ausentes: 0
['Brasil', 'Norte', 'Rondônia', 'Acre', 'Amazonas', 'Roraima', 'Pará', 'Amapá', 'Tocantins', 'Nordeste', 'Maranhão', 'Piauí', 'Ceará', 'Rio Grande do Norte', 'Paraíba', 'Pernambuco', 'Alagoas', 'Sergipe', 'Bahia', 'Sudeste', 'Minas Gerais', 'Espírito Santo', 'Rio de Janeiro', 'São Paulo', 'Sul', 'Paraná', 'Santa Catarina', 'Rio Grande do Sul', 'Centro-Oeste', 'Mato Grosso do Sul', 'Mato Grosso', 'Goiás', 'Distrito Federal']


As contas são subtrações de horas por semana: mulher preta ou parda menos o grupo comparado. Valor positivo é mais tempo. Valor negativo é menos tempo.


In [4]:
def horas_a_mais(lugar, coluna_base):
    linha = indicador_1.loc[indicador_1["uf_regiao"] == lugar].iloc[0]
    diferenca = linha["mulher_preta_parda"] - linha[coluna_base]
    if diferenca > 0:
        sentido = "a mais"
    elif diferenca < 0:
        sentido = "a menos"
    else:
        sentido = "o mesmo tempo"
    print(f"{lugar}: {linha['mulher_preta_parda']:.4f} - {linha[coluna_base]:.4f} = {diferenca:.4f}")
    print(f"Mulheres pretas ou pardas trabalham {abs(diferenca):.1f} h/semana {sentido}.")
    print()
    return diferenca

diff_mg = horas_a_mais("Minas Gerais", "mulher_branca")
diff_brasil = horas_a_mais("Brasil", "homem_branca")
diff_ap = horas_a_mais("Amapá", "mulher_branca")
diff_pr = horas_a_mais("Paraná", "mulher_branca")


Minas Gerais: 23.0758 - 22.1866 = 0.8891
Mulheres pretas ou pardas trabalham 0.9 h/semana a mais.



Brasil: 22.0380 - 11.7051 = 10.3329
Mulheres pretas ou pardas trabalham 10.3 h/semana a mais.

Amapá: 15.9085 - 16.6486 = -0.7401
Mulheres pretas ou pardas trabalham 0.7 h/semana a menos.

Paraná: 19.3654 - 18.8967 = 0.4687
Mulheres pretas ou pardas trabalham 0.5 h/semana a mais.



In [5]:
regioes = ["Norte", "Nordeste", "Sudeste", "Sul", "Centro-Oeste"]
recortes = indicador_1[indicador_1["uf_regiao"].isin(regioes)].copy()
recortes["diferenca_horas"] = recortes["mulher_preta_parda"] - recortes["mulher_branca"]

for _, linha in recortes.iterrows():
    diferenca = linha["diferenca_horas"]
    sentido = "a mais" if diferenca > 0 else "a menos" if diferenca < 0 else "o mesmo tempo"
    print(
        f"{linha['uf_regiao']}: mulheres pretas ou pardas trabalham "
        f"{abs(diferenca):.1f} h/semana {sentido} que mulheres brancas."
    )

recortes[["uf_regiao", "mulher_preta_parda", "mulher_branca", "diferenca_horas"]]

Norte: mulheres pretas ou pardas trabalham 1.2 h/semana a mais que mulheres brancas.
Nordeste: mulheres pretas ou pardas trabalham 1.0 h/semana a mais que mulheres brancas.
Sudeste: mulheres pretas ou pardas trabalham 1.3 h/semana a mais que mulheres brancas.
Sul: mulheres pretas ou pardas trabalham 0.3 h/semana a mais que mulheres brancas.
Centro-Oeste: mulheres pretas ou pardas trabalham 1.1 h/semana a mais que mulheres brancas.


,uf_regiao,mulher_preta_parda,mulher_branca,diferenca_horas
1,Norte,20.534188,19.295780,1.238408
9,Nordeste,23.716716,22.744666,0.972050
19,Sudeste,21.990431,20.732969,1.257462
24,Sul,19.338734,19.012969,0.325765
28,Centro-Oeste,19.331040,18.220683,1.110357
